<a href="https://colab.research.google.com/github/jabes-christian/chatbot-leitor-pdf/blob/main/Automa%C3%A7%C3%A3o_com_llm_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chatbot usando a Interface Gráfica do "**Gradio**"

In [2]:
!pip install gradio langchain langchain-community faiss-cpu pypdf -q

In [3]:
import gradio as gr
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA
import os

In [11]:
# Configuração da API da Groq
GROQ_API_KEY = "SUA_API_KEY_GROQ"
os.environ["OPENAI_API_KEY"] = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model="llama3-70b-8192",
    temperature=0,
)

# Variáveis globais para manter o estado
qa_chain = None

In [6]:
# Função para processar o PDF
def process_pdf(file):
    global qa_chain
    loader = PyPDFLoader(file.name)
    documents = loader.load()

    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(docs, embeddings)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectorstore.as_retriever(),
        return_source_documents=True
    )
    return "PDF carregado e pronto para perguntas!"

# Função para responder perguntas
def responder(pergunta):
    if qa_chain is None:
        return "Por favor, envie um PDF antes de perguntar."
    resposta = qa_chain.invoke({"query": pergunta})
    return resposta["result"]


In [10]:
# Interface Gradio
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Chatbot Leitor de PDF")

    with gr.Row():
        pdf_input = gr.File(label="📄 Envie seu PDF", file_types=[".pdf"])
        carregar_btn = gr.Button("Carregar PDF")

    status_text = gr.Textbox(label="Status")

    pergunta_input = gr.Textbox(label="❓ Faça sua pergunta")
    enviar_btn = gr.Button("Enviar")
    resposta_output = gr.Textbox(label="💬 Resposta do Chatbot")

    carregar_btn.click(process_pdf, inputs=pdf_input, outputs=status_text)
    enviar_btn.click(responder, inputs=pergunta_input, outputs=resposta_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55ced27dc8d3df3659.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
